---
# Vision Transformer Implementation 
---

In this exercise, you will implement a Vision Transformer and train it on CIFAR-10.

---
# Imports
---

Run the code cell below to load the necessary python modules.

In [1]:
import importlib
import tests
import visual
import vit

---
## Vision Transformer
---

Initialize all architectual components of the Vision Transformer Model.

## **Task:**

Implement `ViT.init` in `vit.py`.

**Hints:**
- Use `nn.Linear` and `nn.Sequential` to create the (1) patch projection and (2) the final MLP head for classification outputs.
- Choose `nn.Tanh` as for this specific MLP.
- Use your `create_transformer_encoder` function to create the encoder.
- Your model will fully learn the position embeddings during training. Randomly initialize them using `nn.Parameter`.
- Create the CLS-token tensor using `nn.Parameter(torch.randn(1, 1, model_dim))`.
- Positional enncoding and CLS token: Use `torch.randn` for init.

Rund the code cell below.

In [2]:
importlib.reload(vit)
tests.test_vit_init()

PASS: ViT initialization is correct.


True

---
## ViT Forward Pass
---

If everything is initialized correctly, you can implement the ViT forward pass.

## **Task:** 

Implement `ViT.forward` in `vit.py`.

Hints:
- Use `patchify.create_patch_sequence` to turn the input images into a batch of patch sequences.
- Prepend your CLS-token tensor to all sequences in your batch. Use `torch.Tensor.expand` and `torch.cat`. Your Sequence tensor should have shape `(batch_size, num_patches + 1, model_dim)` afterward.
- Add the position embedding tensor to your sequence tensor using `+`. 
- Use only the final outputs of the CLS tokens for the classification.

Run the code cell below.

In [3]:
importlib.reload(vit)
tests.test_vit_forward()

FAIL: ViT.forward does not prepend the CLS token or add positional embeddings correctly.


False

---
## ViT Training Loop
---

Implement the training loop for your ViT model. Use the provided initialization of `model` because training can take a long time. If you want, you can tune your own architecture and hyperparameters afterwards.  

## **Task:**

Implement `vit.training_step` and `vit.train` in `vit.py`.

Hints:
- Load the data with `data.make_cifar10_loaders`.
- Use `torch.optim.AdamW` as your optimizer and pass it `lr` and `weight_decay`.
- Use `torch.optim.lr_scheduler.CosineAnnealingLR` as your learning rate scheduler. Make sure to pass a value to the `T_max` parameter.
- Use `to(device)` on all relevant tensors to ensure your model and data are on the GPU.
- Choose `nn.CrossEntropyLoss` as your loss function.
- You can add simple logging using `print` to see the training progress. Dont log every single batch, since printing can bottleneck your training speed.
- Use `model.train()` and `model.eval()` in the coresponding phases. Look at `torch.no_grad` to ensure you don't update your model during evaluation.


The model should achieve roughly $77\%$ accuracy after 30 epochs. It should be around $75\%$ at 20 epochs and $67\%$ at 10 epochs if you implemented everything correctly. 

Run the code cells below.

In [4]:
# Adjust these if you want to try to find better hyperparameters.
batch_size = 256
lr = 5e-4
weight_decay = 0.1
# you can lower the epochs to 20 if training takes too long
num_epochs = 30

In [5]:
model, train_loss, test_acc = vit.train(
        batch_size=batch_size,
        lr=lr,
        weight_decay=weight_decay,
        epochs=num_epochs,
    )
visual.show_training_stats(train_loss, test_acc).show()

In [6]:
num_params = sum(p.numel() for p in model.parameters())
print(f"Your ViT model has {num_params:_} parameters.")

Your ViT model has 4_064_842 parameters.


---
# Visualize Attention
---

You can run the cell below to look at some images from the test set and see the final attention weights of the CLS token. If the images look very underwhelming, rerun the cell a couple of times. You should be able to find some images for which you can see meaningful attention weights. 

In [7]:
visual.cifar10_attention(model, n=5)